# 6 - Model testing

Using the output `cv_best_params` from model training, we test our models on the data set starting from 03/17/2025 (Monday).

### Data processing and set-up


We process the data and set up the required packages.

In [1]:
import pandas, numpy, matplotlib, seaborn, sklearn, statsmodels, prophet

print("All packages imported successfully!")

All packages imported successfully!


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
from seaborn import set_style
from sklearn.metrics import mean_squared_error

set_style("whitegrid")

We load the data from `data/arxiv-totals.parquet` and set up our training data set from 01/01/2001 (Monday) to 03/14/2025 (Monday), and testing data set to 03/17/2025 (Monday).

In [3]:
df = pd.read_parquet("../data/arxiv-totals.parquet")

Later we will do the linear regression on the weekdays of the week: we extract and one-hot encode it.

In [4]:
from calendar import day_name

df.reset_index(inplace=True)
df["weekday"] = df["date"].apply(lambda date: day_name[date.weekday()])

one_hot_weekday = (
    pd.get_dummies(df.weekday, dtype=int).drop("Friday", axis=1).iloc[:, [0, 2, 3, 1]]
)
df = df.join(one_hot_weekday)

df.set_index("date", inplace=True)
df.columns = df.columns.astype(str)


# from calendar import day_name

# # Copy the DataFrame to avoid modifying the original
# dg = df

# # Reset index to have 'date' as a column
# dg.reset_index(inplace=True)
# dg["weekday"] = dg["date"].apply(lambda date: day_name[date.weekday()])

# # One-hot encode the 'weekday' column, excluding 'Friday'
# one_hot_weekday = (
#     pd.get_dummies(dg.weekday, dtype=int).drop("Friday", axis=1).iloc[:, [0, 2, 3, 1]]
# )

# # Drop existing weekday columns if they exist
# weekday_cols = ["Monday", "Tuesday", "Wednesday", "Thursday"]
# dg = dg.drop(columns=[col for col in weekday_cols if col in dg.columns])

# # Now join the new one-hot encoded columns
# dg = dg.join(one_hot_weekday)

# dg.set_index("date", inplace=True)
# dg.columns = dg.columns.astype(str)

# dg_train = dg[
#     (dg.index >= pd.Timestamp(2001, 1, 1)) & (dg.index <= pd.Timestamp(2025, 3, 14))
# ]
# dg_test = dg[dg.index >= pd.Timestamp(2025, 3, 17)]

In [5]:
df_train = df[
    (df.index >= pd.Timestamp(2001, 1, 1)) & (df.index <= pd.Timestamp(2025, 3, 14))
]
df_test = df[df.index >= pd.Timestamp(2025, 3, 17)]

In [6]:
print(df.columns)
print(df_train.shape, df_test.shape)
print(df.head())

Index(['hep-th', 'physics.pop-ph', 'math.LO', 'math.FA', 'math.MG', 'cs.CC',
       'math.CO', 'math.PR', 'math.DS', 'cs.GR',
       ...
       'econ.GN', 'eess.AS', 'eess.IV', 'eess.SP', 'q-fin.MF', 'weekday',
       'Monday', 'Tuesday', 'Wednesday', 'Thursday'],
      dtype='object', length=163)
(6315, 163) (20, 163)
            hep-th  physics.pop-ph  math.LO  math.FA  math.MG  cs.CC  math.CO  \
date                                                                            
1986-04-28     1.0             1.0      0.0      0.0      0.0    0.0      0.0   
1988-11-14     1.0             0.0      0.0      0.0      0.0    0.0      0.0   
1989-04-17     0.0             0.0      1.0      0.0      0.0    0.0      0.0   
1989-10-27     0.0             0.0      0.0      3.0      3.0    0.0      0.0   
1989-11-10     0.0             0.0      0.0      1.0      1.0    0.0      0.0   

            math.PR  math.DS  cs.GR  ...  econ.GN  eess.AS  eess.IV  eess.SP  \
date                           

<!-- This indicates that the time series has seasonality, with season of a week. Other categories exhibit similar correlograms, so effective models should likely take weekly seasonality into account (notice that the seasonal parameter should be 5 instead of 7 since the papers are only submitted on business days). Looking at the graphs, there is also a global trend to take into account. -->

We will use [`statsmodels`](https://www.statsmodels.org/stable/index.html) as our choice of time series library (Install the module `statsmodels` by using Anaconda `conda install -c conda-forge statsmodels`). In particular, see [Time Series analysis `tsa`](https://www.statsmodels.org/devel/tsa.html).

In [7]:
## Importing statsmodels to check that we have it installed
import statsmodels as sm

In [8]:
## printing the statsmodels version
print(sm.__version__)

0.14.4


We load the data output from model training.

In [9]:
import json
import pandas as pd

dg = pd.read_csv("cv_results.csv", index_col=0)

# Optionally convert to nested dictionary: category -> model -> score
cv_results_dict = dg.to_dict(orient="index")  # {model: {category: value}}
with open("cv_best_params.json", "r") as f:
    cv_best_params_dict = json.load(f)

### Model testing


For each category and each method, we run the model on `test_size=15` with the best parameters from model tuning.

In [ ]:
# type: ignore

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.api import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
from prophet import Prophet
import numpy as np
import warnings
import json

test_results_dict = {}

warnings.filterwarnings("ignore")

# Load categories
with open("../data/arxiv-categories.json", "r") as f:
    arxiv_categories_descriptions = json.load(f)
categories = sorted([cat["tag"] for cat in arxiv_categories_descriptions])
# categories = sorted(
#     [cat["tag"] for cat in arxiv_categories_descriptions]
#     # We exclude ["q-bio", "cond-mat", "astro-ph"] because they disappeared before our sample starting date.
# )

# Ensure business day frequency
df_train = df_train.asfreq("B")
df_test = df_test.asfreq("B")

# Weekday linear regression model
day_reg = LinearRegression()
day_rmses = np.zeros(5)

for i, category in enumerate(categories, 1):

    print(f"[{i}/{len(categories)}] Testing category: {category}")

    # Prepare train and test data
    y_train = df_train[category].fillna(0)
    y_test = df_test[category].fillna(0)
    train_mean = y_train.mean()

    best_params = cv_best_params_dict[category]
    test_results = {}

    # Dummy
    dummy_value = best_params["Dummy"]["value"]
    dummy_preds = np.full_like(y_test, dummy_value)
    dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_preds))
    test_results["Dummy"] = dummy_rmse / train_mean

    # Weekday linear regression model
    df_tt = df_train.reset_index()
    df_holdout = df_test.reset_index()
    # Fit the linear regression model
    day_reg.fit(
        df_tt[["Monday", "Tuesday", "Wednesday", "Thursday"]],
        df_tt[category],
    )
    day_preds = day_reg.predict(
        df_holdout[["Monday", "Tuesday", "Wednesday", "Thursday"]]
    )
    rmse = np.sqrt(mean_squared_error(df_holdout[category], day_preds))
    test_results["Weekday_Linear"] = rmse / train_mean

    # EST_NCV (manual smoothing)
    try:
        p = best_params["EST_NCV"]
        model = ExponentialSmoothing(
            y_train, trend=p["trend"], seasonal=p["seasonal"], seasonal_periods=5
        ).fit(
            smoothing_level=p["smoothing_level"],
            smoothing_trend=p["smoothing_trend"],
            smoothing_seasonal=p["smoothing_seasonal"],
            optimized=False,
        )
        preds = model.forecast(len(y_test))
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        test_results["EST_NCV"] = rmse / train_mean
    except:
        test_results["EST_NCV"] = np.nan

    # EST_CV (auto smoothing)
    try:
        p = best_params["EST_CV"]
        model = ExponentialSmoothing(
            y_train, trend=p["trend"], seasonal=p["seasonal"], seasonal_periods=5
        ).fit(optimized=True)
        preds = model.forecast(len(y_test))
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        test_results["EST_CV"] = rmse / train_mean
    except:
        test_results["EST_CV"] = np.nan

    # SARIMA_CV (manual seasonal order)
    try:
        seasonal_order = tuple(best_params["SARIMA_CV"]["seasonal_order"])
        model = ARIMA(y_train, order=(0, 0, 0), seasonal_order=seasonal_order)
        fitted = model.fit()
        preds = fitted.forecast(steps=len(y_test))
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        test_results["SARIMA_CV"] = rmse / train_mean
    except:
        test_results["SARIMA_CV"] = np.nan

    # Prophet
    prophet_rmses = []
    prophet_params = best_params["Prophet"]
    df_tt = df_train.tail(150).reset_index()
    df_holdout = df_test.reset_index()
    y_fold = df_tt[category].fillna(0)
    # Ensure date column is in datetime format
    df_tt["date"] = pd.to_datetime(df_tt["date"])
    df_holdout["date"] = pd.to_datetime(df_holdout["date"])
    prophet = Prophet(
        seasonality_mode=prophet_params["seasonality_mode"],
        weekly_seasonality=prophet_params["weekly_seasonality"],
        yearly_seasonality=prophet_params["yearly_seasonality"],
        changepoint_prior_scale=prophet_params["changepoint_prior_scale"],
    )
    # Prepare dataframes for Prophet
    prophet_tt = df_tt[["date", category]].rename(columns={"date": "ds", category: "y"})
    prophet_holdout = df_holdout[["date", category]].rename(
        columns={"date": "ds", category: "y"}
    )
    # Fit Prophet model
    prophet.fit(prophet_tt)
    forecast = prophet.predict(prophet_holdout[["ds"]])
    preds = forecast["yhat"].values
    # Calculate RMSE for Prophet model
    rmse = np.sqrt(mean_squared_error(prophet_holdout["y"], preds))
    prophet_rmses.append(rmse)
    avg_prophet_rmse = np.nanmean(prophet_rmses)
    test_results["Prophet"] = (
        avg_prophet_rmse / train_mean if not np.isnan(avg_prophet_rmse) else np.nan
    )

    # Prophet_Full
    prophet_full_rmses = []
    prophet_params = best_params["Prophet_Full"]
    df_tt = df_train.reset_index()
    df_holdout = df_test.reset_index()
    y_fold = df_tt[category].fillna(0)
    # Ensure date column is in datetime format
    df_tt["date"] = pd.to_datetime(df_tt["date"])
    df_holdout["date"] = pd.to_datetime(df_holdout["date"])
    prophet_full = Prophet(
        seasonality_mode=prophet_params["seasonality_mode"],
        weekly_seasonality=prophet_params["weekly_seasonality"],
        yearly_seasonality=prophet_params["yearly_seasonality"],
        changepoint_prior_scale=prophet_params["changepoint_prior_scale"],
    )
    # Prepare dataframes for Prophet
    prophet_full_tt = df_tt[["date", category]].rename(
        columns={"date": "ds", category: "y"}
    )
    prophet_full_holdout = df_holdout[["date", category]].rename(
        columns={"date": "ds", category: "y"}
    )
    # Fit Prophet model
    prophet_full.fit(prophet_tt)
    forecast_full = prophet_full.predict(prophet_full_holdout[["ds"]])
    preds_full = forecast_full["yhat"].values
    # Calculate RMSE for Prophet model
    rmse_full = np.sqrt(mean_squared_error(prophet_full_holdout["y"], preds_full))
    prophet_full_rmses.append(rmse_full)
    avg_prophet_full_rmse = np.nanmean(prophet_full_rmses)
    test_results["Prophet_Full"] = (
        avg_prophet_full_rmse / train_mean
        if not np.isnan(avg_prophet_full_rmse)
        else np.nan
    )

    # Save test results
    test_results_dict[category] = test_results

[1/155] Testing category: astro-ph.CO


11:50:49 - cmdstanpy - INFO - Chain [1] start processing
11:50:49 - cmdstanpy - INFO - Chain [1] done processing
11:50:49 - cmdstanpy - INFO - Chain [1] start processing
11:50:49 - cmdstanpy - INFO - Chain [1] done processing


[2/155] Testing category: astro-ph.EP


11:50:51 - cmdstanpy - INFO - Chain [1] start processing
11:50:51 - cmdstanpy - INFO - Chain [1] done processing
11:50:51 - cmdstanpy - INFO - Chain [1] start processing
11:50:51 - cmdstanpy - INFO - Chain [1] done processing


[3/155] Testing category: astro-ph.GA


...

11:56:19 - cmdstanpy - INFO - Chain [1] start processing
11:56:19 - cmdstanpy - INFO - Chain [1] done processing
11:56:19 - cmdstanpy - INFO - Chain [1] start processing
11:56:19 - cmdstanpy - INFO - Chain [1] done processing


[154/155] Testing category: stat.OT


11:56:21 - cmdstanpy - INFO - Chain [1] start processing
11:56:21 - cmdstanpy - INFO - Chain [1] done processing
11:56:21 - cmdstanpy - INFO - Chain [1] start processing
11:56:21 - cmdstanpy - INFO - Chain [1] done processing


[155/155] Testing category: stat.TH


11:56:24 - cmdstanpy - INFO - Chain [1] start processing
11:56:24 - cmdstanpy - INFO - Chain [1] done processing
11:56:24 - cmdstanpy - INFO - Chain [1] start processing
11:56:24 - cmdstanpy - INFO - Chain [1] done processing


We save the test results to `test_results.csv`.

In [11]:
test_results_df = pd.DataFrame.from_dict(test_results_dict, orient="index")
test_results_df.to_csv("test_results.csv")